# E16 역할별 헤드 탐색 — 20% 재현 + 동일 평가셋의 20%·80% 비교
**해결책 | 역할별 top-4 → 합집합 → HEAD_LAST 연결 → Logistic Regression**

박세령  / 이화여자대학교 · 지도교수 황의원  

**Colab의 새 런타임에서 위에서 아래로 모두 실행합니다.** 실제 E14의 중립 내부 표현만 입력으로 재사용하고, 헤드 순위·선택·정규화·LR은 각 조건에서 새로 계산합니다. 과거 예측 파일을 읽어 학습을 생략하지 않습니다. 이 배포본에는 실행 출력이 없습니다.

| 패널 | 사용 라벨 | 평가 질문 | 목적 |
|---|---:|---:|---|
| A. `E16_source20_test80` | 122질문 | 490질문 | 발표의 E16 20% 조건 재현 |
| B. `matched_test20`, 20% | 122질문 | 공통 122질문 | 소량 라벨 조건 |
| B. `matched_test20`, 80% | 490질문 | **위와 같은 122질문** | 같은 방법의 큰 라벨 예산 조건 |

모든 패널은 시드 **42–46, 5회 반복**합니다. 80% 인원은 E16 수치 함수의 반올림 규칙으로 490개입니다. 5-fold CV나 disjoint OOF가 아닙니다. 패널 B의 20% 소스는 80% 소스의 부분집합이며, 각 시드에서 패널 A의 소스와도 같습니다. 패널 A에서 평가했던 490개 전체를 80% 학습과 동시에 평가에 쓰지 않습니다.

## 0. 무엇을 유지하고, 무엇을 비교하는가?

**문제상황:** 적은 K/C 라벨로도 유용한 헤드 부분집합을 찾을 수 있는가? 같은 방법에 더 많은 라벨을 주면 얼마나 달라지는가?

**방법:** 발표 시연 파일 `alltology_KC_probe_demo.ipynb`에 내장된 E16 수치 함수를 기준으로 분할, 역할별 스칼라, AUROC 순위, top-4 합집합, LR 설정을 유지합니다. E14 가변-k 분류기나 전체 32헤드·두 위치 결합 방법으로 바꾸지 않습니다.

| 역할 | 순위에 쓰는 스칼라 | 소스에서 사용하는 목표 라벨 |
|---|---|---|
| memory | 질문 위치 HEAD_Q 벡터의 L2 크기 | K |
| context | 마지막 query가 문맥 토큰에 준 attention mass | C |
| conflict | `||HEAD_LAST−HEAD_Q|| / max(||HEAD_Q||, 1e−8)` | `K != C` |

각 헤드의 `max(AUROC, 1−AUROC)`로 역할별 순위를 정하고 동점이면 작은 head ID를 먼저 선택합니다. **세 역할의 상위 4개씩의 합집합**을 사용하므로 최종 헤드는 4–12개입니다. 선택 후 LR에는 해당 헤드들의 **HEAD_LAST 원 벡터**만 연결해 입력합니다. HEAD_Q는 역할별 헤드 탐색용이지 최종 LR의 추가 입력이 아닙니다.

**대조:** 전체 은닉 표현 LR, 같은 개수의 무작위 헤드 LR 20회, 동일 선발 헤드의 probe-free scalar rule. 무작위 LR은 선정 방식만 대조하도록 차원·소스·LR을 맞춥니다. 규칙 점수를 지도학습 판독기의 점수와 구분합니다.

**효과 판단:** 새로 측정한 평균·표준편차·동일 test의 대응 차이로 결론을 작성합니다. 어느 시드나 방법도 test 점수를 보고 교체하지 않습니다. 이미 연구한 p8s 내 재평가이며, 이 재실행만으로 새로운 외부 데이터 일반화나 QA 개선을 입증하지 않습니다.

In [ ]:
# 1. 환경. 새 런타임의 첫 코드 셀로 실행하세요.
import sys, subprocess, importlib.metadata, importlib.util
TARGET_SKLEARN = '1.6.1'  # 발표 E16 수치 함수의 기준 버전
try:
    current = importlib.metadata.version('scikit-learn')
except importlib.metadata.PackageNotFoundError:
    current = None
if current != TARGET_SKLEARN:
    if 'sklearn' in sys.modules:
        raise RuntimeError('sklearn이 이미 로드됐습니다. 런타임을 재시작하고 첫 셀부터 실행하세요.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'scikit-learn=={TARGET_SKLEARN}'])
for module, package in [('numpy','numpy'), ('pandas','pandas'), ('gdown','gdown'),
                        ('threadpoolctl','threadpoolctl'), ('joblib','joblib')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
print('scikit-learn:', importlib.metadata.version('scikit-learn'))


In [ ]:
# 2. 설정: 결과 폴더는 실행마다 새로 생성합니다.
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json, hashlib, math, time, warnings, platform, shutil
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
from IPython.display import display
import joblib

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
PROJECT_ROOT = Path('/content/drive/MyDrive/p8s') if IN_COLAB else Path.cwd() / 'p8s'
WORK = Path('/content/e16_role_inputs') if IN_COLAB else Path.cwd() / 'e16_role_inputs'
RUN_ID = 'E16_ROLE_20_80_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
OUT = PROJECT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
CFG = {
    'protocol': 'E16_ROLE_MATCHED_v1', 'seeds': [42,43,44,45,46],
    'replication_budgets': [0.20],  # 원래 5·10·20% 표도 필요하면 실행 전에 [0.05,0.10,0.20]으로 변경
    'comparison_budgets': [0.20,0.80],
    'top_heads_per_role': 4, 'lr_C': 0.1, 'max_iter': 2000,
    'random_union_reps': 20, 'bootstrap_reps': 5000,
    'result_cache_allowed': False,
    'source_code_sha256': 'ff7da0fe5dd3ce6adc738781e83a14aa876c49467eb45e27ee3f107185cba32b',
}
STATE_NAMES = ['K+C+', 'K+C-', 'K-C-', 'K-C+']
ROLE_NAMES = ('memory', 'context', 'conflict')
FEATURE_SPEC = {
    'memory': 'l2(HEAD_Q)', 'context': 'C_MASS',
    'conflict': 'l2(HEAD_LAST-HEAD_Q)/max(l2(HEAD_Q),1e-8)',
}
ENVIRONMENT = {'python': platform.python_version(),
               **{x: importlib.metadata.version(x) for x in ['numpy','pandas','scipy','scikit-learn','threadpoolctl']}}
FIT_EVENTS, BLOCK_EVENTS = [], []
ACTIVE_BLOCK = None
print('새 결과 폴더:', OUT)
print('이전 예측 재사용: 금지 | 입력 표현 재사용: 허용 | seeds:', CFG['seeds'])


In [ ]:
def canonical_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False,
                                    allow_nan=False).encode()).hexdigest()

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def json_native(x):
    if isinstance(x, dict): return {str(k): json_native(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [json_native(v) for v in x]
    if isinstance(x, np.ndarray): return json_native(x.tolist())
    if isinstance(x, np.generic): return json_native(x.item())
    if isinstance(x, float) and not np.isfinite(x): return None
    return x

def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(json_native(obj), ensure_ascii=False, indent=2,
                              sort_keys=True, allow_nan=False), encoding="utf-8")
    tmp.replace(path)

def atomic_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False); tmp.replace(path)

def state_ids(k, c):
    k, c = np.asarray(k, int), np.asarray(c, int)
    if not set(np.unique(k)).issubset({0, 1}) or not set(np.unique(c)).issubset({0, 1}):
        raise ValueError("K/C must be binary")
    return np.where(k == 1, np.where(c == 1, 0, 1), np.where(c == 1, 3, 2)).astype(int)

def state_to_action(states):
    """A prespecified K/C policy, NOT an oracle of actual per-row intervention gain."""
    y = np.asarray(states, int)
    return np.where(y == 1, "retain", np.where(y == 3, "accept", "noop"))


In [ ]:
# 3. 기존 특징·라벨만 자동 확보합니다. 이전 실험 결과 CSV는 입력으로 받지 않습니다.
SOURCES = {
    'behavior': {'id':'1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr', 'name':'p8s_behavior_v2.csv',
                 'sha256':'fd37b0afd04972e3834ffa5f0af7f362affcfac8e20325117d37d8de15447c1c'},
    'bank': {'id':'1GK0xi217u57Wz_kn8ycg5eSlPgjsaFNy', 'name':'L13_neutral_head_bank_all_rows.npz',
             'sha256':'4ad4a040388d1e45995be7aadc7b91f6dfd1f4091b4fd452da8b4a9036b83802'},
}
def obtain_inputs(work, roots):
    work = Path(work); work.mkdir(parents=True, exist_ok=True)
    paths = {}
    for key, spec in SOURCES.items():
        target = work / spec['name']
        candidates = [target] + [Path(root) / spec['name'] for root in roots]
        local = next((p for p in candidates if p.is_file() and file_sha256(p)==spec['sha256']), None)
        if local is not None:
            if local.resolve()!=target.resolve(): shutil.copy2(local, target)
        else:
            import gdown
            tmp = target.with_name(target.name+'.download')
            fetched = gdown.download(id=spec['id'], output=str(tmp), quiet=False)
            if fetched is None or not tmp.is_file():
                raise RuntimeError(f'{spec["name"]} 접근 실패. E14 입력이 있는 Drive 계정을 연결하세요.')
            if file_sha256(tmp)!=spec['sha256']:
                raise ValueError('원본 입력 해시 불일치. 다른 데이터로 대체하지 않습니다: '+spec['name'])
            tmp.replace(target)
        if file_sha256(target)!=spec['sha256']: raise ValueError('잘못된 원본 입력: '+key)
        paths[key]=target
        print('[INPUT VERIFIED]', key, spec['name'])
    return paths

PATHS = obtain_inputs(WORK, [PROJECT_ROOT/'E14_L13_head_5fold_v1', PROJECT_ROOT,
                            Path.cwd(), Path('/mnt/data')])
f = pd.read_csv(PATHS['behavior']).reset_index(drop=True)
k = f.kside_corr.map({'K+':1,'K-':0,'K−':0})
c = f.ctx_label.map({'C+':1,'C-':0,'C−':0})
if k.isna().any() or c.isna().any() or f[['question','ctx_text']].isna().any().any():
    raise ValueError('필수 입력·라벨에 결측 또는 알 수 없는 표기가 있습니다.')
INPUTS = f[['question','ctx_text']].copy()
LABELS = pd.DataFrame({'k':k.astype(int), 'c':c.astype(int)})
LABELS['state'] = state_ids(LABELS.k, LABELS.c)
QID = pd.factorize(INPUTS.question, sort=True)[0]
assert len(INPUTS)==1224 and INPUTS.question.nunique()==612
assert INPUTS.groupby('question').size().eq(2).all()
check = LABELS.assign(question=INPUTS.question).groupby('question')
assert check.k.nunique().eq(1).all() and check.c.nunique().eq(2).all()
assert LABELS.state.value_counts().reindex(range(4)).tolist()==[362,362,250,250]
with np.load(PATHS['bank'], allow_pickle=False) as z:
    assert np.array_equal(z['rows'], np.arange(len(INPUTS)))
    BANK = {key:z[key].astype(np.float32) for key in ['BLOCK_LAST','HEAD_LAST','HEAD_Q','C_MASS']}
assert BANK['HEAD_LAST'].shape == BANK['HEAD_Q'].shape == (1224,32,128)
assert BANK['BLOCK_LAST'].shape == (1224,4096) and BANK['C_MASS'].shape == (1224,32)
assert all(np.isfinite(x).all() for x in BANK.values())
# 원본 정답 문자열·생성 답변 등은 학습 함수에 전달하지 않습니다.
del f, k, c, check
atomic_json(OUT/'ENVIRONMENT.json', ENVIRONMENT)
atomic_json(OUT/'SOURCE_HASHES.json', {key:file_sha256(p) for key,p in PATHS.items()})
print('입력 검증 완료:', INPUTS.question.nunique(), '질문 /', len(INPUTS), '행')


## 4. 질문 단위 분할과 라벨 예산

E16의 분할 함수와 `floor(N×비율+0.5)` 계산을 유지합니다. 정렬한 고유 질문을 시드로 섞고 앞부분을 source로 고릅니다. K/C 라벨을 보고 좋은 표본을 다시 뽑지 않습니다. 동일 질문의 C± 두 행은 항상 함께 이동합니다.

각 시드의 순열을 `앞 122 / 다음 368 / 끝 122`로 보면, 패널 A는 앞 122로 학습하고 뒤 490으로 평가합니다. 패널 B는 앞 122 또는 앞 490으로 학습하고 끝 122로만 평가합니다. **패널 B의 평가 질문은 두 라벨 조건 모두의 학습·선택에서 제외**됩니다.

라벨 예산은 각 학습 조건의 접근량입니다. 5개 시드 전체의 라벨 합집합, 평가용 라벨, 논문 전체의 주석 비용이 20%라는 뜻은 아닙니다.

In [ ]:
def budget_count(n_questions, fraction):
    return int(math.floor(n_questions * float(fraction) + 0.5))

def make_question_split(inputs, seed, pool_fraction, budgets):
    """Split by question text, without looking at K, C, state, or correctness.
    All budgets share one held-out set. Smaller sources are nested; unused
    source-pool rows are not used for normalization or calibration.
    """
    forbidden = {"k", "c", "state", "gold", "aliases", "em", "ctx_label"}
    if forbidden.intersection(inputs.columns):
        raise AssertionError("Splitting accepts a label-free input table only")
    q = np.array(sorted(inputs.question.astype(str).unique()), dtype=object)
    rng = np.random.default_rng(int(seed)); order = rng.permutation(q)
    n_pool = budget_count(len(q), pool_fraction)
    if not 1 < n_pool < len(q): raise ValueError("Invalid source-pool size")
    source_pool, test_questions = order[:n_pool], order[n_pool:]
    test = np.flatnonzero(inputs.question.isin(set(test_questions)).to_numpy())
    splits = {}
    for b in budgets:
        nq = budget_count(len(q), b)
        if not 2 <= nq <= n_pool: raise ValueError(f"budget {b} not within source pool")
        source_q = set(order[:nq])
        source = np.flatnonzero(inputs.question.isin(source_q).to_numpy())
        assert not set(source).intersection(test)
        assert not set(inputs.iloc[source].question).intersection(inputs.iloc[test].question)
        unused = np.flatnonzero(~inputs.index.isin(np.concatenate([source, test])))
        splits[float(b)] = dict(source=source, test=test.copy(), unused=unused,
                                n_source_questions=nq, n_test_questions=len(test_questions))
    return splits

class BudgetLabelAccess:
    """Enforces label access inside one run; final scoring is deliberately separate."""
    def __init__(self, labels, allowed_rows):
        self._labels = labels
        self.allowed = set(map(int, allowed_rows)); self.accessed = set(); self.log = []
    def read(self, rows, purpose):
        rows = np.asarray(rows, int)
        if not set(rows).issubset(self.allowed):
            raise AssertionError("Attempted to read labels outside the registered budget")
        self.accessed.update(map(int, rows))
        self.log.append({"purpose": str(purpose), "n_rows": len(rows)})
        return self._labels.iloc[rows][["k", "c", "state"]].copy()
    def audit(self):
        return {"allowed_rows": sorted(self.allowed), "accessed_rows": sorted(self.accessed),
                "n_unique_labeled_rows": len(self.accessed), "log": self.log}


In [ ]:
# 모든 split을 성능 계산 전에 확정하고 저장합니다.
PLAN = []
for seed in CFG['seeds']:
    panels = [
        ('E16_source20_test80', 0.20, CFG['replication_budgets']),
        ('matched_test20', 0.80, CFG['comparison_budgets']),
    ]
    for panel, pool, budgets in panels:
        specs = make_question_split(INPUTS, seed, pool, budgets)
        for budget, spec in specs.items():
            PLAN.append({'panel':panel, 'seed':seed, 'budget':budget, **spec})
for seed in CFG['seeds']:
    a = next(s for s in PLAN if s['panel']=='E16_source20_test80' and s['seed']==seed and s['budget']==.20)
    lo = next(s for s in PLAN if s['panel']=='matched_test20' and s['seed']==seed and s['budget']==.20)
    hi = next(s for s in PLAN if s['panel']=='matched_test20' and s['seed']==seed and s['budget']==.80)
    assert np.array_equal(a['source'], lo['source'])
    assert set(lo['source']).issubset(hi['source'])
    assert np.array_equal(lo['test'], hi['test'])
    assert set(hi['source']).isdisjoint(lo['test'])
    assert (a['n_source_questions'], a['n_test_questions'])==(122,490)
    assert (lo['n_source_questions'],hi['n_source_questions'],lo['n_test_questions'])==(122,490,122)
manifest=[]
for s in PLAN:
    role=np.full(len(INPUTS),'unused',object)
    role[s['source']]='source'; role[s['test']]='test'
    manifest.append(pd.DataFrame({'panel':s['panel'],'seed':s['seed'],'budget':s['budget'],
        'row':np.arange(len(INPUTS)), 'qid':QID, 'question':INPUTS.question, 'role':role}))
SPLIT_MANIFEST=pd.concat(manifest,ignore_index=True)
atomic_csv(OUT/'SPLIT_MANIFEST.csv',SPLIT_MANIFEST)
atomic_json(OUT/'ANALYSIS_PLAN.json', {
    'run_id':RUN_ID,'config':CFG,'environment':ENVIRONMENT,'splits':PLAN,
    'claim':'E16 low-label role discovery and paired label-budget comparison',
    'primary_method':'diagnostic_same_roleheads_LR',
    'primary_contrast':'mean_seed[F1(matched20)-F1(matched80)]',
    'repetitions':'5 repeated question holdouts; not disjoint OOF',
    'selection':'source-only scalar oriented-AUC ranking; no test tuning',
    'external_preregistration':False,
})
PLAN_SHA=file_sha256(OUT/'ANALYSIS_PLAN.json')
print('계획 고정:',PLAN_SHA)
display(pd.DataFrame([{k:s[k] for k in ['panel','seed','budget','n_source_questions','n_test_questions']} for s in PLAN]))


## 5. 역할별 헤드를 고르는 실제 코드

아래 함수들은 발표 시연 파일의 E16 수치 함수에서 가져왔습니다. **source의 스칼라와 source의 K/C 라벨만** 받습니다. memory·context·conflict별 top-4를 찾고 합집합을 만듭니다. 이 단계는 감독학습 기반 선택이지만 별도의 head별 LR을 훈련하는 단계는 아닙니다.

probe-free 대조는 같은 source에서 계산한 부호·중앙값/MAD·중앙값 임계값을 고정합니다. 최종 4-state 규칙은 memory와 context의 두 축으로 구성되며, conflict 점수는 role 선발 및 참고 action 계산에 사용됩니다. 실제 QA 답변 생성은 수행하지 않습니다.

In [ ]:
def raw_features(bank):
    hq, hl = bank["HEAD_Q"].astype(np.float32), bank["HEAD_LAST"].astype(np.float32)
    qnorm = np.linalg.norm(hq, axis=-1)
    features = {
        "memory": qnorm,
        "context": bank["C_MASS"].astype(np.float32),
        "conflict": np.linalg.norm(hl - hq, axis=-1) / np.maximum(qnorm, 1e-8),
    }
    for k, x in features.items():
        if x.ndim != 2 or not np.isfinite(x).all(): raise ValueError(f"Bad raw feature {k}")
    return features


In [ ]:
def fit_role_rule(raw_source, source_labels, source_rows, source_questions,
                  top_k=4, random_head_seed=None):
    """E14 role selection + E16 median two-axis state decoder.
    Labels are used for AUC ranking/sign, not for quantile-boundary optimization.
    No previous E14 head IDs, probe weights or thresholds are reused.
    """
    yk = source_labels.k.to_numpy(int); yc = source_labels.c.to_numpy(int)
    if set(yk) != {0, 1} or set(yc) != {0, 1}:
        raise ValueError("소스 budget에 K+/K-, C+/C-가 모두 있어야 합니다. seed를 성능으로 고르지 마세요.")
    targets = {"memory": yk, "context": yc, "conflict": (yk != yc).astype(int)}
    rank_rows = []; role_locks = {}; norm = {}
    for ri, role in enumerate(ROLE_NAMES):
        x = np.asarray(raw_source[role], np.float32)
        if len(x) != len(source_labels): raise ValueError("Source/label length mismatch")
        oriented, signs = [], []
        for head in range(x.shape[1]):
            auc = float(roc_auc_score(targets[role], x[:, head]))
            sign = 1.0 if auc >= 0.5 else -1.0
            oriented.append(max(auc, 1 - auc)); signs.append(sign)
        order = sorted(range(x.shape[1]), key=lambda h: (-oriented[h], h))
        if random_head_seed is None: selected = order[:top_k]
        else:
            # Head count and source-only sign orientation are matched.
            selected = np.random.default_rng(random_head_seed + ri * 100).choice(
                x.shape[1], size=top_k, replace=False).tolist()
        role_locks[role] = [{"head": int(h), "sign": signs[h]} for h in selected]
        signed = np.column_stack([x[:, h] * signs[h] for h in selected])
        med = np.median(signed, axis=0)
        mad = np.median(np.abs(signed - med), axis=0)
        scale = np.maximum(1.4826 * mad, 1e-6)
        norm[role] = {"median": med.tolist(), "scale": scale.tolist()}
        for h in range(x.shape[1]):
            rank_rows.append(dict(role=role, head=h, oriented_auc=oriented[h],
                                  sign=signs[h], selected=h in selected))
    policy = {
        "schema": "E16_E14_role_median_v1", "read_layer_zero_based": 13,
        "source_rows": list(map(int, source_rows)),
        "source_questions": sorted(set(map(str, source_questions))),
        "role_lock": role_locks, "normalizer": norm, "feature_spec": FEATURE_SPEC,
        "thresholds": {}, "uses_probe_at_runtime": False,
        "source_KC_labels_used_for_head_selection_and_sign": True,
        "labels_used_for_threshold_optimization": False,
        "target_stats_or_labels_used_for_adaptation": False,
        "top_heads_per_role_fixed_in_advance": int(top_k),
        "fourstate_decoder": "K=memory>=source_q50; C=context>=source_q50; then Cartesian state",
        "state_threshold_is_new_E16_extension_not_E14_F1": True,
    }
    s = role_scores(raw_source, policy)
    policy["thresholds"] = {
        "memory_cut": float(np.quantile(s["memory"], 0.50)),
        "context_cut": float(np.quantile(s["context"], 0.50)),
        "memory_hi": float(np.quantile(s["memory"], 0.70)),
        "memory_lo": float(np.quantile(s["memory"], 0.30)),
        "context_hi": float(np.quantile(s["context"], 0.65)),
        "conflict_hi": float(np.quantile(s["conflict"], 0.70)),
    }
    return policy, pd.DataFrame(rank_rows)


In [ ]:
def role_scores(raw, policy):
    ans = {}
    for role in ROLE_NAMES:
        vals = np.column_stack([np.asarray(raw[role])[:, h["head"]] * h["sign"]
                               for h in policy["role_lock"][role]])
        norm = policy["normalizer"][role]
        ans[role] = np.median((vals - np.asarray(norm["median"])) /
                              np.asarray(norm["scale"]), axis=1)
    return ans

def predict_role_rule(raw, policy):
    """Input: unlabeled activations + frozen JSON. All samples receive a state."""
    scores = role_scores(raw, policy); t = policy["thresholds"]
    k = (scores["memory"] >= t["memory_cut"]).astype(int)
    c = (scores["context"] >= t["context_cut"]).astype(int)
    states = state_ids(k, c)
    e14_actions = np.where((scores["memory"] >= t["memory_hi"]) &
                          (scores["conflict"] >= t["conflict_hi"]), "retain",
                   np.where((scores["memory"] <= t["memory_lo"]) &
                            (scores["context"] >= t["context_hi"]), "accept", "noop"))
    return pd.DataFrame({"pred_state": states, "pred_K": k, "pred_C": c,
        "K_score": scores["memory"], "C_score": scores["context"],
        "conflict_score": scores["conflict"], "state_action": state_to_action(states),
        "e14_action": e14_actions})

def union_heads(policy):
    return sorted({d["head"] for v in policy["role_lock"].values() for d in v})


In [ ]:
RAW = raw_features(BANK)
HEAD_LAST = BANK['HEAD_LAST']
BLOCK_LAST = BANK['BLOCK_LAST']

# 학습 이력: fit이 정상 종료한 경우만 기록합니다.
def train_lr(x_source, y_source, seed, method, rep=0):
    if ACTIVE_BLOCK is None: raise RuntimeError('학습 블록 시작 전입니다.')
    if set(np.unique(y_source)) != {0,1,2,3}:
        raise ValueError('소스에 4-state가 모두 없습니다. 실패한 시드를 교체하지 않습니다.')
    model=make_pipeline(StandardScaler(), LogisticRegression(
        C=CFG['lr_C'], max_iter=CFG['max_iter'], solver='lbfgs', random_state=int(seed)))
    started=time.perf_counter()
    with warnings.catch_warnings(), threadpool_limits(limits=2):
        warnings.simplefilter('error',ConvergenceWarning)
        model.fit(x_source,y_source)
    assert np.array_equal(model.classes_,np.arange(4))
    np.testing.assert_allclose(model[0].mean_, x_source.mean(axis=0,dtype=np.float64),rtol=1e-7,atol=1e-7)
    FIT_EVENTS.append({'run_id':RUN_ID,'block':ACTIVE_BLOCK,'method':method,'rep':rep,
        'source_rows':len(x_source),'features':x_source.shape[1],
        'elapsed_seconds':time.perf_counter()-started,'status':'fit_completed',
        'used_result_cache':False})
    return model

def predict_lr(model,x):
    with threadpool_limits(limits=2): p=model.predict_proba(x)
    pred=p.argmax(1)
    out=pd.DataFrame({'pred_state':pred,'pred_K':np.isin(pred,[0,1]).astype(int),
        'pred_C':np.isin(pred,[0,3]).astype(int),
        'K_score':p[:,[0,1]].sum(1),'C_score':p[:,[0,3]].sum(1)})
    for j in range(4): out[f'p{j}']=p[:,j]
    return out

def begin_block(folder):
    global ACTIVE_BLOCK
    if ACTIVE_BLOCK is not None: raise RuntimeError('이전 학습이 미완료입니다. 설정 셀부터 새 실행을 시작하세요.')
    folder.mkdir(parents=False,exist_ok=False)
    ACTIVE_BLOCK=folder.name
    print('[TRAIN START]',ACTIVE_BLOCK,flush=True)
    return len(FIT_EVENTS),time.perf_counter()

def end_block(folder,token):
    global ACTIVE_BLOCK
    start,t0=token; expected=2+CFG['random_union_reps']
    actual=len(FIT_EVENTS)-start
    if actual!=expected: raise RuntimeError(f'학습 누락: {actual}/{expected}')
    event={'block':ACTIVE_BLOCK,'run_id':RUN_ID,'actual_fits':actual,'expected_fits':expected,
        'seconds':time.perf_counter()-t0,'result_reuse':0,'status':'fresh_completed'}
    atomic_json(folder/'FIT_CALLS.json',FIT_EVENTS[start:])
    atomic_json(folder/'COMPLETE.json',event)
    BLOCK_EVENTS.append(event)
    ACTIVE_BLOCK=None
    print(f'[TRAIN DONE] {folder.name} | fits={actual}/{expected} | {event["seconds"]:.1f}s | result reuse=0',flush=True)


## 6. 새 학습·선택·예측

각 조건에서 source 라벨을 허용한 뒤 역할별 헤드를 새로 고르고, source 전체에 최종 LR을 학습합니다. LR의 설정은 미리 고정되어 있고 test에서 선택하지 않습니다. 정규화는 source에서만 fit합니다.

**무작위 헤드 LR:** 선택한 합집합과 같은 개수의 헤드를 32개 중 균일하게 뽑아 동일 차원의 LR을 학습합니다. 20회 결과를 모두 저장하고 평균냅니다. 우수한 무작위 조합만 골라 보고하지 않습니다.

선발·학습 및 모든 대조군의 예측을 완료하고 파일에 저장한 다음에야 평가 함수가 test의 정답을 읽습니다. 패널 A의 점수를 보고 패널 B의 설정을 바꾸지 않도록, **모든 패널의 학습·예측을 먼저 끝낸 뒤 일괄 채점**합니다.

In [ ]:
def run_one(spec):
    panel,seed,budget=spec['panel'],spec['seed'],spec['budget']
    src,te=spec['source'],spec['test']
    folder=OUT/f'{panel}_seed{seed}_b{int(budget*100):02d}'
    token=begin_block(folder)
    access=BudgetLabelAccess(LABELS,src)
    labels=access.read(src,'role ranking, signs, scalar calibration, LR fitting')
    source_raw={key:x[src] for key,x in RAW.items()}
    policy,ranks=fit_role_rule(source_raw,labels,src,INPUTS.iloc[src].question,top_k=CFG['top_heads_per_role'])
    heads=union_heads(policy)
    assert 4<=len(heads)<=12
    y=labels.state.to_numpy(int)
    policies={'role_heads':heads,'n_heads':len(heads),'n_features':128*len(heads),
              'source_questions':spec['n_source_questions'],'test_questions':spec['n_test_questions']}
    atomic_json(folder/'ROLE_POLICY.json',policy)
    atomic_json(folder/'SELECTED_HEADS.json',policies)
    atomic_csv(folder/'ROLE_RANKING.csv',ranks)
    model_main=train_lr(HEAD_LAST[src][:,heads,:].reshape(len(src),-1),y,seed,'diagnostic_same_roleheads_LR')
    model_full=train_lr(BLOCK_LAST[src],y,seed,'diagnostic_fullhidden_LR')
    random_models=[]; random_manifest=[]
    for rep in range(CFG['random_union_reps']):
        rng=np.random.default_rng(np.random.SeedSequence([16080,int(seed),int(budget*100),rep]))
        selected=sorted(rng.choice(32,size=len(heads),replace=False).tolist())
        model=train_lr(HEAD_LAST[src][:,selected,:].reshape(len(src),-1),y,seed,'random_union_LR',rep)
        random_models.append((rep,selected,model))
        random_manifest.append({'rep':rep,'heads':selected,'n_heads':len(selected)})
    # 모든 설정과 학습을 완료한 뒤 test 특징을 읽습니다.
    predictions=[]
    def attach(pred,method,rep=0):
        p=pred.copy()
        p['row']=te; p['qid']=QID[te]
        p['panel']=panel; p['seed']=seed; p['budget']=budget
        p['method']=method;p['rep']=rep
        predictions.append(p)
    attach(predict_lr(model_main,HEAD_LAST[te][:,heads,:].reshape(len(te),-1)),'diagnostic_same_roleheads_LR')
    attach(predict_lr(model_full,BLOCK_LAST[te]),'diagnostic_fullhidden_LR')
    attach(predict_role_rule({key:x[te] for key,x in RAW.items()},policy),'runtime_rule')
    for rep,hs,model in random_models:
        attach(predict_lr(model,HEAD_LAST[te][:,hs,:].reshape(len(te),-1)),'random_union_LR',rep)
    unscored=pd.concat(predictions,ignore_index=True)
    if any(c in unscored for c in ['true_state','gold','ctx_label','kside_corr']): raise AssertionError('정답은 예측 파일에 아직 넣지 않습니다.')
    atomic_csv(folder/'PREDICTIONS_UNSCORED.csv',unscored)
    audit=access.audit()
    assert set(audit['accessed_rows'])==set(src)
    audit.update({'source_questions':spec['n_source_questions'], 'test_questions':spec['n_test_questions'],
                  'source_test_question_overlap':0,'test_labels_used_for_training_or_selection':False})
    atomic_json(folder/'LABEL_ACCESS.json',audit)
    atomic_json(folder/'RANDOM_HEADS.json',random_manifest)
    # 이 노트북에서 학습한 모델만 저장합니다. 외부 pickle을 불러오지 않습니다.
    joblib.dump({'model':model_main,'heads':heads,'state_order':STATE_NAMES,'source_hash':file_sha256(PATHS['bank'])},folder/'ROLEHEAD_LR.joblib')
    end_block(folder,token)
    return {'spec':spec,'folder':folder,'policy':policy,'heads':heads,'predictions':unscored}

# 기존 완료 폴더가 있으면 읽거나 덮어쓰지 않고 오류로 중단합니다.
if any(OUT.glob('*_seed*_b*')):
    raise FileExistsError('학습 단계가 이미 시작된 폴더입니다. 설정 셀부터 재실행해 새 폴더를 만드세요.')
RUNS=[]
try:
    for spec in PLAN: RUNS.append(run_one(spec))
except Exception as exc:
    atomic_json(OUT/'FAILED.json',{'block':ACTIVE_BLOCK,'error':repr(exc),'completed_blocks':len(BLOCK_EVENTS)})
    raise
expected=len(PLAN)*(2+CFG['random_union_reps'])
assert ACTIVE_BLOCK is None and len(BLOCK_EVENTS)==len(PLAN) and len(FIT_EVENTS)==expected
atomic_json(OUT/'FRESH_EXECUTION_AUDIT.json',{'run_id':RUN_ID,'status':'fresh_training_verified',
    'blocks':len(BLOCK_EVENTS),'expected_blocks':len(PLAN),'fit_calls':len(FIT_EVENTS),'expected_fit_calls':expected,
    'reused_prediction_blocks':0,'input_features_reused':True,'plan_sha256':PLAN_SHA})
atomic_csv(OUT/'FIT_CALLS.csv',pd.DataFrame(FIT_EVENTS))
print(f'[FRESH VERIFIED] fit={len(FIT_EVENTS)}/{expected}; result reuse=0')


In [ ]:
def macro_from_cm(cm):
    cm = np.asarray(cm, float)
    tp = np.diagonal(cm, axis1=-2, axis2=-1)
    den = cm.sum(-1) + cm.sum(-2)
    return np.mean(np.divide(2 * tp, den, out=np.zeros_like(tp), where=den > 0), axis=-1)

def metrics_state(y_true, predictions):
    y = np.asarray(y_true, int); pred = predictions.pred_state.to_numpy(int)
    cm = confusion_matrix(y, pred, labels=np.arange(4))
    support, predsupport = cm.sum(1), cm.sum(0); tp = np.diag(cm).astype(float)
    recalls = np.divide(tp, support, out=np.zeros(4), where=support > 0)
    f1 = np.divide(2 * tp, support + predsupport, out=np.zeros(4), where=(support + predsupport) > 0)
    out = {"n": len(y), "four_macro_F1": float(macro_from_cm(cm)),
           "four_BAcc_fixed4": float(recalls.mean()), "four_accuracy": float(np.mean(y == pred)),
           "state_prediction_coverage": 1.0}
    for j, name in enumerate(STATE_NAMES):
        out[f"F1_{name}"] = float(f1[j]); out[f"n_{name}"] = int(support[j])
    for name, positive in [("K", [0, 1]), ("C", [0, 3])]:
        truth = np.isin(y, positive).astype(int)
        hard = predictions[f"pred_{name}"].to_numpy(int)
        cm2 = confusion_matrix(truth, hard, labels=[0, 1])
        rs = cm2.sum(1)
        out[f"{name}_BAcc"] = float(np.mean(np.divide(np.diag(cm2), rs,
                                                       out=np.zeros(2), where=rs > 0)))
        out[f"{name}_AUROC"] = float(roc_auc_score(truth, predictions[f"{name}_score"])) if len(set(truth)) == 2 else np.nan
    for a in ["state_action", "e14_action"]:
        if a in predictions:
            out[a + "_intervention_rate"] = float(predictions[a].ne("noop").mean())
    return out, cm


In [ ]:
# 7. 최종 채점. 이 셀부터 test 정답을 예측과 대조합니다.
SCORED=[]; METRIC_ROWS=[]; PER_STATE=[]; SELECTED=[]
for result in RUNS:
    s=result['spec']; pred=result['predictions'].copy()
    pred['true_state']=LABELS.state.to_numpy()[pred.row.to_numpy(int)]
    SCORED.append(pred)
    for (method,rep),g in pred.groupby(['method','rep'],sort=False):
        g=g.sort_values('row')
        assert np.array_equal(g.row.to_numpy(),s['test'])
        met,cm=metrics_state(g.true_state.to_numpy(),g)
        METRIC_ROWS.append({'panel':s['panel'],'seed':s['seed'],'budget':s['budget'],
            'method':method,'rep':rep,'n_heads':len(result['heads']) if method!='diagnostic_fullhidden_LR' else np.nan,
            'uses_probe_at_runtime':method!='runtime_rule',**met})
        support=cm.sum(1); predicted=cm.sum(0); tp=np.diag(cm).astype(float)
        precision=np.divide(tp,predicted,out=np.zeros(4),where=predicted>0)
        recall=np.divide(tp,support,out=np.zeros(4),where=support>0)
        for j,name in enumerate(STATE_NAMES):
            PER_STATE.append({'panel':s['panel'],'seed':s['seed'],'budget':s['budget'],'method':method,'rep':rep,
                'state':name,'precision':precision[j],'recall':recall[j],'F1':met['F1_'+name],'support':int(support[j])})
        if method!='random_union_LR':
            pd.DataFrame(cm,index=STATE_NAMES,columns=STATE_NAMES).to_csv(result['folder']/f'CONFUSION_{method}.csv')
    atomic_csv(result['folder']/'PREDICTIONS_SCORED.csv',pred)
    SELECTED.append({'panel':s['panel'],'seed':s['seed'],'budget':s['budget'],
        'n_heads':len(result['heads']),'vector_dimension':len(result['heads'])*128,
        'heads':json.dumps(result['heads']),
        **{r:json.dumps([x['head'] for x in result['policy']['role_lock'][r]]) for r in ROLE_NAMES}})
SCORED=pd.concat(SCORED,ignore_index=True)
METRICS=pd.DataFrame(METRIC_ROWS)
atomic_csv(OUT/'ALL_PREDICTIONS.csv',SCORED)
atomic_csv(OUT/'METRICS_BY_RUN.csv',METRICS)
atomic_csv(OUT/'PER_STATE_METRICS.csv',pd.DataFrame(PER_STATE))
atomic_csv(OUT/'HEAD_SELECTIONS.csv',pd.DataFrame(SELECTED))
# 무작위 20회는 시드 내부에서 먼저 평균합니다. 100개의 독립 실험으로 세지 않습니다.
score_cols=['four_macro_F1','four_BAcc_fixed4','four_accuracy','K_AUROC','C_AUROC']
SEED_METRICS=METRICS.groupby(['panel','budget','method','seed'],as_index=False)[score_cols].mean()
summary_rows=[]
for (panel,budget,method),g in SEED_METRICS.groupby(['panel','budget','method'],sort=False):
    summary_rows.append({'panel':panel,'budget':budget,'method':method,'n_seeds':len(g),
        'F1_mean':g.four_macro_F1.mean(),'F1_seed_sd':g.four_macro_F1.std(ddof=1),
        'K_AUROC_mean':g.K_AUROC.mean(),'C_AUROC_mean':g.C_AUROC.mean(),
        'uses_probe_at_runtime':method!='runtime_rule'})
SUMMARY=pd.DataFrame(summary_rows)
atomic_csv(OUT/'METRICS_BY_SEED.csv',SEED_METRICS)
atomic_csv(OUT/'PAPER_SUMMARY.csv',SUMMARY)
display(SUMMARY.round(6))


## 8. 같은 평가 질문에서 20%−80% 차이 계산

주 지표는 시드별 **Macro-F1을 먼저 계산한 뒤 평균한 값**입니다. `mean_seed[F1(20%)−F1(80%)]`를 보고합니다. 시드 예측을 앙상블하지 않고, 겹치는 test를 하나의 disjoint OOF로 합치지 않습니다.

조건부 신뢰구간은 612개 질문 ID를 복원추출한 공통 가중치를 모든 시드·조건에 동시에 적용해 계산합니다. 각 시드에서 원래 test에 해당하는 질문만 사용하며 두 C행은 함께 움직입니다. 이미 학습한 판독기와 고정된 split에 조건부인 구간으로, 재학습 불확실성·새 데이터 일반화를 모두 포괄하지 않습니다. test를 보며 허용 손실을 정해 비열등성을 선언하지 않습니다.

In [ ]:
def paired_questions_ci(scored, panel, method_a, budget_a, method_b, budget_b, reps=5000):
    # 각 seed에 동일한 test rows와 ground truth가 있는지 확인합니다.
    n_questions=INPUTS.question.nunique(); parts_a=[];parts_b=[];points=[]
    for seed in CFG['seeds']:
        def take(method,budget):
            g=scored[(scored.panel==panel)&(scored.method==method)&(scored.budget==budget)&(scored.seed==seed)]
            if g.rep.nunique()!=1: raise ValueError('이 비교 함수는 seed당 한 판독기의 예측만 받습니다.')
            return g.sort_values('row')
        a,b=take(method_a,budget_a),take(method_b,budget_b)
        assert len(a)>0 and np.array_equal(a.row,b.row) and np.array_equal(a.true_state,b.true_state)
        qa=np.zeros((n_questions,16),float);qb=np.zeros_like(qa)
        np.add.at(qa,(a.qid.to_numpy(int),4*a.true_state.to_numpy(int)+a.pred_state.to_numpy(int)),1)
        np.add.at(qb,(b.qid.to_numpy(int),4*b.true_state.to_numpy(int)+b.pred_state.to_numpy(int)),1)
        parts_a.append(qa);parts_b.append(qb)
        points.append(float(macro_from_cm(qa.sum(0).reshape(4,4))-macro_from_cm(qb.sum(0).reshape(4,4))))
    A=np.stack(parts_a);B=np.stack(parts_b)
    rng=np.random.default_rng(1616080);dist=[]
    with threadpool_limits(limits=2):
        for start in range(0,reps,100):
            count=min(100,reps-start)
            w=rng.multinomial(n_questions,np.full(n_questions,1/n_questions),size=count)
            ca=np.einsum('bn,snk->bsk',w,A,optimize=True).reshape(count,len(A),4,4)
            cb=np.einsum('bn,snk->bsk',w,B,optimize=True).reshape(count,len(A),4,4)
            dist.extend((macro_from_cm(ca)-macro_from_cm(cb)).mean(1).tolist())
    lo,hi=np.quantile(dist,[.025,.975])
    return {'panel':panel,'A':method_a,'budget_A':budget_a,'B':method_b,'budget_B':budget_b,
        'delta_F1':float(np.mean(points)),'ci95_low':float(lo),'ci95_high':float(hi),
        'seed_delta_sd':float(np.std(points,ddof=1)),'bootstrap_reps':reps,
        'CI_scope':'conditional_on_fitted_readers_and_fixed_holdouts; common_question_weights'}

contrasts=[paired_questions_ci(SCORED,'matched_test20','diagnostic_same_roleheads_LR',.20,
                              'diagnostic_same_roleheads_LR',.80,CFG['bootstrap_reps'])]
for panel,budgets in [('E16_source20_test80',[.20]),('matched_test20',[.20,.80])]:
    for b in budgets:
        contrasts.append(paired_questions_ci(SCORED,panel,'diagnostic_same_roleheads_LR',b,
                         'diagnostic_fullhidden_LR',b,CFG['bootstrap_reps']))
PAIRED=pd.DataFrame(contrasts)
atomic_csv(OUT/'PAIRED_COMPARISONS.csv',PAIRED)
# 이 negative delta는 낮은 예산 쪽의 감소를 뜻합니다. 값 자체를 숨기지 않습니다.
display(PAIRED[['panel','A','budget_A','B','budget_B','delta_F1','ci95_low','ci95_high']].round(6))


In [ ]:
# 9. 논문에 옮길 표와 문장: 이번 실행값만 사용합니다.
def summary_for(panel,budget,method='diagnostic_same_roleheads_LR'):
    g=SUMMARY[(SUMMARY.panel==panel)&(SUMMARY.budget==budget)&(SUMMARY.method==method)]
    assert len(g)==1
    return g.iloc[0]
a=summary_for('E16_source20_test80',.20)
lo=summary_for('matched_test20',.20);hi=summary_for('matched_test20',.80)
contrast=PAIRED.iloc[0]
lines=[
    '# E16 역할별 헤드 탐색: 실제 실행 결과', '',
    f'Run: {RUN_ID}',f'환경: {json.dumps(ENVIRONMENT,ensure_ascii=False)}','',
    '## A. 발표 E16 조건',
    f'전체 612개 질문 중 122개 질문의 라벨로 역할별 헤드 탐색과 LR 학습을 수행하고 나머지 490개 질문에서 평가하였다. '
    f'시드 42–46의 5회 반복 Macro-F1은 {a.F1_mean:.4f}±{a.F1_seed_sd:.4f}였다.','',
    '## B. 동일 평가 질문에서 라벨 예산 비교',
    f'각 시드에서 공통 평가 질문 122개를 남긴 뒤, 동일한 역할별 선발·판독 절차에 122개 또는 490개 질문의 라벨을 사용하였다. '
    f'20% 조건의 Macro-F1은 {lo.F1_mean:.4f}±{lo.F1_seed_sd:.4f}, '
    f'80% 조건은 {hi.F1_mean:.4f}±{hi.F1_seed_sd:.4f}였다. '
    f'대응 차이(20%−80%)는 {contrast.delta_F1:+.4f}, '
    f'질문 단위 조건부 95% 신뢰구간은 [{contrast.ci95_low:+.4f}, {contrast.ci95_high:+.4f}]였다.','',
    '5회 반복은 5-fold OOF가 아니다. A와 B의 평가 질문 수는 다르며 두 패널의 점수를 섞어 대응 차이를 계산하지 않는다.',
    '지도학습 LR 점수와 probe-free scalar rule 점수는 별도 보고한다. QA 생성·steering 성능은 이 노트북에서 측정하지 않는다.',
    '동등성·비열등성 자동 판정은 하지 않는다. p8s는 이미 탐색된 연구 자료이며 독립적인 새 데이터 검증은 아니다.',
]
REPORT='\n'.join(lines)
(OUT/'PAPER_RESULTS_KO.md').write_text(REPORT,encoding='utf-8')
# 공식 방법 기술에서 각 역할의 스칼라·AUROC 방향 보정·정렬·LR 설정을 명시합니다.
methods=('L13에서 질문 위치의 헤드 벡터 크기, 문맥 attention mass, 질문 위치 대비 마지막 위치의 상대 변화량을 각각 memory, context, conflict 후보 신호로 사용하였다. '
         '소스 K, C, K≠C 라벨에 대한 방향 보정 AUROC를 기준으로 각 역할의 상위 4개 헤드를 선택하고 합집합을 구하였다. '
         '선택된 각 헤드의 128차원 HEAD_LAST 벡터를 연결하여 StandardScaler와 Logistic Regression(C=0.1, lbfgs, max_iter=2000)을 소스 전체에 학습하였다. '
         '평가 질문은 모든 선택 및 학습에서 제외하였다.')
(OUT/'PAPER_METHOD_KO.txt').write_text(methods,encoding='utf-8')
atomic_json(OUT/'FINAL_STATUS.json',{'status':'complete','run_id':RUN_ID,
    'fit_calls':len(FIT_EVENTS),'blocks':len(BLOCK_EVENTS),'source_plan_sha256':PLAN_SHA,
    'reported_as':'repeated_holdout_not_disjoint_OOF','reused_predictions':0})
# 결과 파일의 해시를 남깁니다. 모델·라벨 접근·선정 헤드까지 추적할 수 있습니다.
paths=sorted(p for p in OUT.rglob('*') if p.is_file() and p.name!='OUTPUT_HASHES.json')
atomic_json(OUT/'OUTPUT_HASHES.json',{str(p.relative_to(OUT)):file_sha256(p) for p in paths})
print(REPORT)
print('결과 폴더:',OUT)


In [ ]:
# 10. 이 실행의 결과만 ZIP으로 묶습니다. 입력 feature bank는 포함하지 않습니다.
zip_path=Path(shutil.make_archive(str(OUT.parent/(RUN_ID+'_results')),'zip',root_dir=OUT))
print('완료:',zip_path)
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))


## 코드 출처·검증 범위

- 연구 구성: 제공한 발표자료의 E16 source20/test80 및 역할별 top-4 합집합 설명.
- 수치 구현: `alltology_KC_probe_demo.ipynb`의 E16 numerical core. `make_question_split`, `raw_features`, `fit_role_rule`, `role_scores`, `predict_role_rule`, `union_heads`, `metrics_state`의 수치 정의를 유지했습니다.
- 추가 부분: 동일 test의 20%·80% 패널, 개수 맞춘 무작위 헤드 LR 대조, fresh-only 학습 감사, 조건부 대응 비교 및 논문용 자동 집계.
- 분할은 정답을 보지 않는 질문별 무작위 순열입니다. 전체 32개 헤드의 **스칼라를 선발 단계에서 조사하는 것**과 전체 32개 헤드의 **벡터를 최종 판독기에 모두 넣는 것**은 다릅니다. 주 방법은 선택된 합집합의 HEAD_LAST 벡터만 사용합니다.
**제작 검증:** 실제 612질문/1,224행 입력으로 기본 구성 15조건과 330회의 LR fit, 모든 채점 및 대응 부트스트랩을 완료했습니다. 로컬 검증 환경은 Python 3.13.5 / scikit-learn 1.8.0 / NumPy 2.3.5 / pandas 2.2.3 / SciPy 1.17.0입니다. 배포 노트북은 발표 E16 기준인 scikit-learn 1.6.1을 설치하므로, 로컬 검증값을 그 버전의 실행 결과로 표시하지 않습니다. 역할·분할 수치 함수의 AST 일치, source/test 질문 분리, 예산 외 라벨 접근 차단, 동일 source의 공통 행 예측 일치, top-4 합집합 차원, F1 독립 재계산, 결과 재사용 없음까지 확인했습니다. Colab의 파일 다운로드·Drive 인증은 로컬 검증 범위 밖입니다. 이번 배포본의 모든 code-cell 출력과 실행 횟수는 비어 있습니다.